# Early Detection of Collective Psychological Pressure

> **Aggregate, population-level language signals only. No diagnosis. No individual-level analysis.**

This notebook detects stress-related *linguistic signals* in public online text and reports only aggregate patterns
(cells with n < K_MIN are suppressed). It does not assess, profile, rank or re-identify any individual.
Findings are phrased as "stress-related language in the sampled communities shifted", never as "the population became stressed".

**Runs on any Windows laptop: just Run All.**
- **0.0** installs what is missing and picks the torch build for this laptop. It trains on the laptop's NVIDIA GPU (CUDA) if
  it has one, otherwise on the CPU.
- **Nothing runs twice.** Every step that computes or writes something runs once. Later Run Alls print
  `SKIP <step>: already done on <machine> at <time>` and load the saved result. Training starts automatically for whatever is not
  trained yet, and an interrupted seed resumes. To redo a step on purpose, add its number to `FORCE_RERUN` in cell 0.1.
- **What ran where, and when:** `trained_models/HANDOFF.md` (one row per step, with machine, device, time and duration).
- **Raw datasets are optional.** Steps 1–2 need manually downloaded raw data (SAD, Mendeley, SenticNet, Zenodo). On a laptop
  without it they print `NOT RUN …`, and Step 3 still works: it only needs Dreaddit, which downloads from Hugging Face.

Cells marked `# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<` train models.

## Step 0: Setup, configuration and device

In [ ]:
# Step 0.0: SETUP for THIS Windows laptop (yours or anyone's). Installs the libraries into this kernel's Python and makes
# sure torch can use the laptop's NVIDIA GPU. Safe to re-run: pip runs only when something is missing, at the wrong
# version, or when torch cannot use the GPU (e.g. a CPU-only torch is installed on a laptop that has an NVIDIA GPU).
# Torch build per laptop (picked from nvidia-smi):
#   NVIDIA GPU    -> CUDA build: 12.1 (driver >= 528), 11.8 (older driver), 12.8 for RTX 50-series (needs torch 2.7.1,
#                    UNVERIFIED with the pinned transformers 4.46.3: tell the assistant if training errors there)
#   no NVIDIA GPU -> CPU build (training works, but takes hours per seed)
import importlib.metadata as md
import json, os, platform, re, subprocess, sys, tempfile
from pathlib import Path

if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f"Python {sys.version.split()[0]} at {sys.executable}: torch / pyspark need Python 3.10-3.12 "
                       "(3.11 recommended). In VS Code pick a 3.11 environment (Select Kernel > Python Environments).")

_root = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (c / "code" / "config.yaml").is_file())
_req = _root / "requirements.txt"
PT = "https://download.pytorch.org/whl/"


def _pins(req_file):
    """[(name, version)] for every `name==version` line (extras like psycopg[binary] are stripped)."""
    hits = (re.match(r"^\s*([A-Za-z0-9_.\-]+)(?:\[[^\]]*\])?==([^\s;#]+)", l) for l in req_file.read_text(encoding="utf-8").splitlines())
    return [m.groups() for m in hits if m]


def _missing_or_wrong(req_file, skip=("torch",)):
    """(name, wanted, installed) for every pin that is not satisfied (torch is handled separately, per laptop)."""
    bad = []
    for name, want in _pins(req_file):
        if name.lower() in skip:
            continue
        try:
            have = md.version(name)
        except md.PackageNotFoundError:
            have = None
        if have is None or have.split("+")[0] != want:
            bad.append((name, want, have))
    return bad


def _pip(*args):
    print("$ python -m pip", " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", *args], text=True, capture_output=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode:
        raise RuntimeError(f"pip failed (exit {r.returncode}); see the output above")


def _nvidia_gpu():
    """First NVIDIA GPU via nvidia-smi (works before torch is installed), or None."""
    for fields in ("name,compute_cap,memory.total,driver_version", "name,memory.total,driver_version"):
        try:
            r = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True, timeout=30)
        except (OSError, subprocess.SubprocessError):
            return None
        if r.returncode == 0 and r.stdout.strip():
            v = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
            if len(v) == 4:
                return {"name": v[0], "cap": float(v[1]), "vram_gb": round(float(v[2]) / 1024, 1), "driver": v[3]}
            return {"name": v[0], "cap": None, "vram_gb": round(float(v[1]) / 1024, 1), "driver": v[2]}
    return None


def _torch_plan(gpu):
    """(torch version, wheel index URL, needs a working CUDA GPU?, why) for this laptop."""
    if gpu is None:
        return "2.5.1", PT + "cpu", False, "no NVIDIA GPU found (nvidia-smi): CPU build"
    drv = float(".".join(gpu["driver"].split(".")[:2]))
    if gpu["cap"] is not None and gpu["cap"] < 5.0:
        return "2.5.1", PT + "cpu", False, f"{gpu['name']} (compute {gpu['cap']}) is too old for current torch: CPU build"
    if gpu["cap"] is not None and gpu["cap"] >= 10.0:
        if drv < 570:
            print(f"  ! driver {gpu['driver']} is too old for {gpu['name']}: update the NVIDIA driver to >= 570")
        return "2.7.1", PT + "cu128", True, f"{gpu['name']} (compute {gpu['cap']}, RTX 50 / Blackwell) needs CUDA 12.8 wheels"
    if drv >= 528.33:
        return "2.5.1", PT + "cu121", True, f"{gpu['name']} (compute {gpu['cap']}), driver {gpu['driver']}: CUDA 12.1 build"
    if drv < 452.39:
        print(f"  ! driver {gpu['driver']} is very old: update the NVIDIA driver, then re-run 0.0")
    return "2.5.1", PT + "cu118", True, f"{gpu['name']}, driver {gpu['driver']} (< CUDA 12 minimum): CUDA 11.8 build"


_PROBE = """
import json, torch
r = {"version": torch.__version__, "cuda": torch.version.cuda, "available": torch.cuda.is_available(), "works": False}
if r["available"]:
    try:
        x = torch.randn(256, 256, device="cuda"); float((x @ x).sum())     # a real kernel launch, not just a flag
        r.update(works=True, device=torch.cuda.get_device_name(0))
    except Exception as e:
        r["error"] = str(e)[:300]
print(json.dumps(r))
"""


def _probe_torch():
    r = subprocess.run([sys.executable, "-c", _PROBE], capture_output=True, text=True)   # fresh process: sees new wheels
    return json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else {"error": r.stderr[-300:]}


print(f"Python {sys.version.split()[0]} | {platform.platform()} | {sys.executable}")
GPU = _nvidia_gpu()
TORCH_VERSION, TORCH_INDEX, WANT_GPU, why = _torch_plan(GPU)
print("NVIDIA GPU:", f"{GPU['name']} | {GPU['vram_gb']} GB | compute {GPU['cap']} | driver {GPU['driver']}" if GPU else "none")
print(f"torch plan: {TORCH_VERSION} from {TORCH_INDEX} ({why})")
_imported_before = [m for m in ("torch", "transformers", "numpy", "pandas", "pyarrow", "sklearn") if m in sys.modules]

# 1) everything except torch (torch line removed so pip never swaps the right torch build for another one)
_todo = _missing_or_wrong(_req)
if _todo:
    print(f"{len(_todo)} package(s) to install/fix: " + ", ".join(f"{n} (have {h or 'none'}, want {w})" for n, w, h in _todo))
    _tmp = Path(tempfile.gettempdir()) / "stress_signals_requirements_no_torch.txt"
    _tmp.write_text("\n".join(l for l in _req.read_text(encoding="utf-8").splitlines()
                              if not re.match(r"^\s*torch==", l)), encoding="utf-8")
    _pip("install", "-r", str(_tmp))
else:
    print(f"all {len(_pins(_req)) - 1} other pinned packages already installed")

# 2) torch: right version AND, if this laptop has a supported NVIDIA GPU, that GPU really runs a computation
T = _probe_torch()
_ok = T.get("version", "").split("+")[0] == TORCH_VERSION and (not WANT_GPU or T.get("works"))
if not _ok:
    print(f"torch needs (re)installing: have {T.get('version', 'none')} (CUDA {T.get('cuda')}, GPU works {T.get('works')})"
          + (f" | error: {T['error']}" if T.get("error") else ""))
    _pip("install", "--force-reinstall", f"torch=={TORCH_VERSION}", "--index-url", TORCH_INDEX)
    T = _probe_torch()

# 3) the project package itself (editable, so code edits apply)
try:
    md.version("stress_signals")
except md.PackageNotFoundError:
    _pip("install", "-e", str(_root / "code"), "--no-deps")

print(f"\ntorch {T.get('version')} | CUDA build {T.get('cuda')} | GPU runs computations {T.get('works')}")
if T.get("works"):
    print(f"TRAINING DEVICE: {T['device']} (cuda, {GPU['vram_gb']} GB)")
elif WANT_GPU:
    print(f"!!! GPU FOUND BUT NOT USABLE BY TORCH: {T.get('error', 'not available')}. Update the GPU driver and re-run 0.0; "
          "until then training runs on the CPU (hours per seed).")
else:
    print("TRAINING DEVICE: CPU (no supported NVIDIA GPU on this laptop). Training works but takes hours per seed.")
if (_todo or not _ok) and _imported_before:
    print(f"\n>>> RESTART THE KERNEL now (already imported before the install: {_imported_before}), then Run All again. <<<")

In [ ]:
# Step 0.1: bootstrap (safe to re-run). Imports the package, loads + validates config, seeds, versions.
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "code" / "config.yaml").is_file():
            return c
    raise FileNotFoundError("code/config.yaml not found above " + str(start))

PROJECT_ROOT = _find_root(Path.cwd().resolve())   # folder holding code/ and data/; all config paths resolve against it
SRC = str(PROJECT_ROOT / "code")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from stress_signals import ETHICS_BANNER, __version__, get_logger, load_config, set_seed
from stress_signals import handoff as H
from stress_signals.utils import device_label, library_versions, models_root, raw_data_status

# Re-run steps on purpose (normally empty): e.g. {"3C.1", "3D.1"} or {"2.5"}. Training is never forced: to retrain a seed,
# move its folder trained_models/stress/runs/<model>/seed<k> somewhere else first.
FORCE_RERUN = set()

CFG = load_config(PROJECT_ROOT / "code" / "config.yaml")
LOG = get_logger("stress_signals", CFG["logging"]["level"], CFG["_paths"]["logs"] / "pipeline.log")
SEED = CFG["primary_seed"]
set_seed(SEED, deterministic=CFG["deterministic"])
RAW = raw_data_status(CFG)
RAW_OK = all(RAW.values())
RAW_WHY = (f"raw datasets missing on this laptop: {[k for k, v in RAW.items() if not v]} (manual downloads, see README). "
           "Step 3 does not need them.")

print("=" * 88)
print(ETHICS_BANNER)
print("=" * 88)
print(f"stress_signals {__version__} | project root: {PROJECT_ROOT}")
print(f"config: {CFG['_config_file']} | seeds: {CFG['seeds']} (primary {SEED}) | K_MIN: {CFG['privacy']['k_min']}")
print(f"models + handoff log: {models_root(CFG)} | device: {device_label()}")
print(f"raw datasets present: {RAW}" + ("" if RAW_OK else "  -> Steps 1-2 limited to Dreaddit"))
if FORCE_RERUN:
    print("FORCE_RERUN:", sorted(FORCE_RERUN))
print("library versions:")
for name, ver in library_versions().items():
    print(f"  {name:<13} {ver if ver else 'NOT INSTALLED'}")

In [ ]:
# Step 0.2: environment check (read-only): the device training and inference will use on THIS laptop.
from stress_signals.utils import detect_environment

ENV = detect_environment()
for key in ("python", "platform", "torch", "torch_cuda_build", "cuda_available", "gpu_name",
            "gpu_capability", "vram_total_gb", "vram_free_gb", "java"):
    print(f"{key:<18} {ENV[key]}")
print(f"\nTRAINING DEVICE: {device_label()}")
for w in ENV["warnings"]:
    print("  !", w)

## Step 1: Acquire and audit all datasets

Read-only inspection: no model weights are created or updated. Outputs: `data/processed/_cache/audit_<name>.json|.md`,
`data/outputs/reports/data_audit.md`, `data/outputs/manifests/step1_audit_manifest.json`. No raw text is written to them.

**Kaggle credentials (GoEmotions Kaggle copy, optional):** kaggle.com → *Settings* → *API* → *Create New Token* downloads
`kaggle.json`. Save it as `%USERPROFILE%\.kaggle\kaggle.json`, or set the `KAGGLE_USERNAME` / `KAGGLE_KEY` environment variables. Never commit it. Without credentials the
audit records the Kaggle copy as `unavailable` and uses the official files plus the HF fallback.

Needs the manually downloaded raw datasets (see README). Without them these cells print `NOT RUN …`.

In [ ]:
# Step 1.1: Zenodo header check (reads only the header row of teaching_post_features_tfidf_256.csv)
from stress_signals import audit as A
if H.ready(RAW_OK, "1.1", RAW_WHY):
    head = A.zenodo_header(CFG, "teaching", "post")
    print("columns:", head["n_columns"])
    print("non-feature columns:", head["non_feature_columns"])
    for k in ("text_column_candidates", "author_columns", "date_columns", "id_columns"):
        print(f"{k:<24}", head[k] or "NONE")

In [ ]:
# Step 1.2: run all audits ONCE (~2 min). Later Run Alls load the saved audit summaries instead.
import json

def _saved_audits():
    return {p.stem.removeprefix("audit_"): json.loads(p.read_text(encoding="utf-8"))
            for p in sorted(CFG["_paths"]["interim"].glob("audit_*.json")) if p.stem != "audit_senticnet_handcheck"}

if H.ready(RAW_OK, "1.2", RAW_WHY):
    results = H.run_step(CFG, "1.2/audits", "Audit all datasets", lambda: A.run(CFG),
                         done_marker=CFG["_paths"]["manifests"] / "step1_audit_manifest.json",
                         skip_result=_saved_audits, force="1.2" in FORCE_RERUN)
    for name, res in results.items():
        print(f"{name:<14}", "ERROR " + res["error"] if "error" in res else "ok", "| flags:", res.get("flags", []))
    print("report:", CFG["_paths"]["reports"] / "data_audit.md")

In [ ]:
# Step 1.3: SenticNet eyeball view: NOTEBOOK VIEW ONLY (scrubbed, truncated, 3 rows per file x label). Not saved anywhere.
import pandas as pd
from stress_signals.privacy import scrub_text
if H.ready(RAW_OK, "1.3", RAW_WHY):
    pd.set_option("display.max_colwidth", 220)
    for name, df in A.load_senticnet(CFG).items():
        tcol, lcol = A.senticnet_text_label_cols(df)
        view = df.groupby(lcol, group_keys=False)[list(df.columns)].apply(lambda g: g.sample(min(3, len(g)), random_state=SEED))
        view = view.assign(text=view[tcol].astype(str).map(lambda t: scrub_text(t, CFG)[:220]))[[lcol, "text"]]
        print(f"--- {name} (text column={tcol!r}, label column={lcol!r})")
        display(view)

In [ ]:
# Step 1.4: create the hand-check sheet (~50 rows stratified by file x label) if it does not exist (never overwritten).
# Open it, fill `human_label` with 0/1 using the definition "the text expresses stress" (blank if unsure), save it.
if H.ready(RAW_OK, "1.4", RAW_WHY):
    SHEET = A.make_senticnet_handcheck(CFG)
    print(SHEET)

In [ ]:
# Step 1.5: score the hand-check, then rebuild the report + manifest. Runs once per version of the sheet: it runs again
# only after you edit (fill in) the sheet.
from stress_signals.utils import sha256_file
if H.ready(RAW_OK, "1.5", RAW_WHY):
    def _score():
        out = A.score_senticnet_handcheck(CFG)
        A.audit_senticnet(CFG)
        A.build_report(CFG)
        A.write_audit_manifest(CFG)
        return out
    HC = H.run_step(CFG, f"1.5/handcheck/{sha256_file(SHEET)[:12]}", "Score the SenticNet hand-check", _score,
                    skip_result=lambda: json.loads((CFG["_paths"]["interim"] / "audit_senticnet_handcheck.json").read_text(encoding="utf-8")),
                    force="1.5" in FORCE_RERUN)
    print(HC)

## Step 2: Preprocessing, unified schema, splits

No model weights are created or updated here. Code: `code/stress_signals/preprocess.py`; tests: `code/tests/test_preprocess.py`.

**Unified schema** (first 8 columns of every `data/processed/*.parquet`): `record_id` (sha256 of source + native id),
`source`, `community`, `created_at` (UTC), `text_clean`, `n_chars`, `n_tokens_est` (regex word+punctuation count, *not* model
tokens), `lang`. Label/split columns follow. No author, username or URL fields exist anywhere.

**Decisions applied (approved in Step 1):**
- Dreaddit: exact-duplicate train texts dropped. If the copies disagree on the label, the whole group is dropped. Train texts that also appear in test are dropped. Official test = FINAL TEST.
  Train/validation comes from `StratifiedGroupKFold` on `post_id`. This is the stratified version of GroupShuffleSplit.
- GoEmotions: official files only, with the official train/dev/test splits (dev is renamed validation). Kaggle copy dropped.
- Zenodo pilot: only `subreddit, date, post` are read. `author` and the feature columns are never read. Text is deduplicated by hash across all files.
  The pre window really is 2018-11-01 to 2019-11-01. `created_at` is day-resolution only.
- SenticNet: validation-only parquet. Its labels are automated. Twitter_Non-Advert is deduplicated against Twitter_Full,
  rows that contain Dreaddit segments are removed, and duplicate groups with conflicting labels are dropped.
- SAD: cleaned into the unified schema. **No split here.** Step 5 owns the stressor splits and the Reddit gold set.

**Filters:** the language and minimum-length filters apply to unlabelled corpora only. Labelled benchmark rows, including official
test rows, are never dropped by them. `lang` is still recorded for them.
**Language ID: lingua-language-detector.** It is deterministic and offline (models ship in the wheel, which matters for Spark workers),
and its authors report better short-text accuracy than langdetect/langid/fastText (our check: UNVERIFIED).
**Chunking:** sentence-aligned windows of at most `max_length` tokens with a `stride`-token overlap. The bound is re-checked by re-tokenising.
**Aggregation:** the default is the *mean* of calibrated chunk probabilities. The max grows with the number of chunks, so long-post
communities (r/relationships) would look systematically more stress-laden than short-post ones. That would bias the aggregates this project reports.
The mean is length-neutral; `max` stays available for sensitivity analysis.

**On a laptop without the raw datasets** only Dreaddit is built (downloaded from Hugging Face). Its fixed split file is in git and is verified, never regenerated.

In [ ]:
# Step 2.1: cleaning demo on SYNTHETIC strings (no user text)
from collections import Counter
from stress_signals import preprocess as P

demo = [
    "OMG &amp;#x200B; **I'm SO stressed** about [rent](https://example.com/x) <url> 😭 ask u/someone or mail a.b@example.org",
    "> quoted\n\n# Heading\n* bullet\n[removed]",
    "Call (555) 123-4567 about the 2019 2020 2021 taxes...",
]
st = Counter()
for s in demo:
    c = P.clean_text(s, CFG, st)
    print(repr(c))
    print("   lexical:", P.clean_text_lexical(c, CFG, already_clean=True))
    assert P.clean_text(c, CFG) == c        # idempotent
print("edit counts:", dict(st))

In [ ]:
# Step 2.2: build the processed datasets + fixed splits + manifest ONCE. All datasets when the raw data is on this laptop,
# otherwise Dreaddit only (downloaded from Hugging Face). Zenodo takes ~10-20 min the first time (resumable cache).
# Saved splits are NEVER regenerated (P.run(CFG, force_splits=True) would).
_which = None if RAW_OK else ["dreaddit"]
_marker = (CFG["_paths"]["manifests"] / "step2_preprocess_manifest.json") if RAW_OK else (CFG["_paths"]["processed"] / "dreaddit.parquet")

def _saved_step2():
    m = json.loads((CFG["_paths"]["manifests"] / "step2_preprocess_manifest.json").read_text(encoding="utf-8"))
    return m.get("extra", {}).get("datasets", {})

RES2 = H.run_step(CFG, "2.2/processed/" + ("all" if RAW_OK else "dreaddit"), "Build processed datasets + splits",
                  lambda: P.run(CFG, only=_which, force="2.2" in FORCE_RERUN), done_marker=_marker,
                  skip_result=_saved_step2, force="2.2" in FORCE_RERUN)
for name, r in RES2.items():
    print(f"{name:<11} final_rows={r['final_rows']}")
print("report:", CFG["_paths"]["reports"] / "preprocess_report.md")

In [ ]:
# Step 2.3: split checks (aggregate counts only; datasets that are not on this laptop are left out)
dre = pd.read_parquet(CFG["_paths"]["processed"] / "dreaddit.parquet")
print("Dreaddit split x label:\n", pd.crosstab(dre["split"], dre["label"], margins=True))
print("post_id overlap between splits (must all be 0):", P.group_overlap(dre))
print("official test rows all in FINAL TEST:", set(dre.loc[dre.official_split == "test", "split"]) == {"test"})
if (CFG["_paths"]["processed"] / "goemotions.parquet").exists():
    goe = pd.read_parquet(CFG["_paths"]["processed"] / "goemotions.parquet", columns=["split", "labels"])
    print("GoEmotions split sizes:", goe["split"].value_counts().to_dict())
for name in ("sad", "zenodo_pilot", "senticnet_validation"):
    p = CFG["_paths"]["processed"] / f"{name}.parquet"
    if p.exists():
        df = pd.read_parquet(p, columns=["community", "lang", "n_tokens_est"])
        print(f"{name}: rows={len(df)}, lang={df['lang'].fillna('und').value_counts().head(3).to_dict()}, "
              f"median n_tokens_est={df['n_tokens_est'].median()}")

In [ ]:
# Step 2.4: chunking check on the Zenodo pilot ONCE: counts only, seeded sample, tokenizer of the stress model
def _chunking_check():
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(CFG["models"]["stress"]["base_model"], use_fast=True)
    ML, STRIDE, MAXC = CFG["models"]["stress"]["max_length"], CFG["chunking"]["stride"], CFG["chunking"]["max_chunks_per_doc"]
    zp = pd.read_parquet(CFG["_paths"]["processed"] / "zenodo_pilot.parquet", columns=["community", "text_clean"])
    sample = pd.concat([g.sample(min(len(g), 300), random_state=SEED) for _, g in zp.groupby("community")])
    rows = []
    for comm, t in zip(sample["community"], sample["text_clean"]):
        ch = P.chunk_text(t, tok, ML, STRIDE, MAXC)
        assert all(c.n_tokens <= ML for c in ch)
        rows.append((comm, ch[0].n_chunks_total, max(c.n_tokens for c in ch)))
    cs = pd.DataFrame(rows, columns=["community", "n_chunks", "max_chunk_tokens"])
    print(f"window={ML}, overlap={STRIDE}, max chunks kept={MAXC}; max tokens seen in any chunk: {cs.max_chunk_tokens.max()}")
    print(cs.groupby("community")["n_chunks"].describe(percentiles=[.5, .9, .99]).round(2))
    print("share of records needing >1 chunk:", round(float((cs.n_chunks > 1).mean()), 3),
          "| share with > max_chunks (subsampled):", round(float((cs.n_chunks > MAXC).mean()), 4))
    print("aggregation example (mean of chunk probabilities):", P.aggregate_chunk_scores([0.2, 0.7, 0.6]))

if H.ready((CFG["_paths"]["processed"] / "zenodo_pilot.parquet").exists(), "2.4", RAW_WHY):
    H.run_step(CFG, "2.4/chunking_check", "Chunking check on the Zenodo pilot", _chunking_check, force="2.4" in FORCE_RERUN)

In [ ]:
# Step 2.5: unit tests (cleaning idempotence, no PII left, chunk bound, no post_id overlap, split protection, ...).
# Runs once per version of the code: it runs again only after code/stress_signals, code/tests or config.yaml changed.
# A failing test stops Run All here on purpose (fix it before training).
import os, subprocess
from stress_signals.utils import code_fingerprint

def _run_tests():
    r = subprocess.run([sys.executable, "-m", "pytest", str(PROJECT_ROOT / "code" / "tests"), "-q", "-p", "no:cacheprovider",
                        "--rootdir", str(PROJECT_ROOT)], cwd=str(PROJECT_ROOT / "code"), capture_output=True, text=True,
                       env={**os.environ, "PYTHONPATH": SRC, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout[-3000:], r.stderr[-2000:])
    if r.returncode:
        raise AssertionError("unit tests failed (see the output above)")

H.run_step(CFG, f"2.5/tests/{code_fingerprint(PROJECT_ROOT)}", "Unit tests for this code version", _run_tests,
           force="2.5" in FORCE_RERUN)

## Step 3: Stress-language classifier on Dreaddit

Code: `code/stress_signals/stress_model.py` (training, inference, shortcut checks, bundle), `code/stress_signals/metrics.py`
(metrics, bootstrap, calibration) and `code/stress_signals/handoff.py` (handoff log, run-once guard).

**Device.** Training and inference use the laptop's NVIDIA GPU (CUDA) if it has one, otherwise the CPU (cell 0.2 shows which).
Cell 3.0c sizes the batch to the GPU and keeps the effective batch at 16 everywhere, so results are comparable between laptops.
The GPU trains in fp16; the CPU in fp32.

**Run All.** The baseline (3A.1) and every seed (3B.1) train automatically if they are not done yet. A seed interrupted mid-training
resumes from its last epoch on the next Run All. Evaluation (3C–3F) waits until every seed exists. The optional leave-one-subreddit-out
trainings (3E.2) run only with `RUN_LOSO = True`. Everything is saved under `trained_models/`. Epoch checkpoints go to `data/outputs/checkpoints/`.

**Decisions.**
- Input is `text_clean` (the Step 2 cleaning of Dreaddit `text`) only. The LIWC/DAL/syntax/social columns are dropped explicitly.
- Model selection uses validation `f1_stress`. The test set is never used for selection, calibration or thresholds.
- `confidence` sample weights are off by default; the docstring of `confidence_weights` explains why.
- The deployable model is the seed with the best validation F1(stress). Temperature and threshold are fitted on validation.
  The threshold is searched only in 0.40-0.60, maximising validation F1(stress). The tuned threshold is **applied only if the validation gain over 0.5 is clear**:
  gain >= 0.01 and the 95% CI of the paired cluster-bootstrap gain is above 0. Otherwise 0.5 is used. Test metrics are reported at **both** 0.5 and the
  tuned threshold, once (guarded file).

In [ ]:
# Step 3.0a: settings, transformers argument names, data guard (no training)
from stress_signals import stress_model as SM, metrics as MX

MODEL_CHOICE = CFG["stress_model"]["model_choice"]     # "roberta-base" | "distilroberta-base" | "deberta-v3-base"
S3 = SM.settings(CFG, MODEL_CHOICE)
compat = SM.hf_compat()
print(f"model: {S3['model_choice']} -> {S3['hf_id']} | max_len {S3['max_length']} | lr {S3['learning_rate']} | "
      f"epochs {S3['epochs']} | warmup {S3['warmup_ratio']} | seeds {S3['seeds']}")
print(f"transformers {compat['transformers']}: eval key = {compat['eval_strategy_key']!r}, "
      f"Trainer tokenizer arg = {compat['tokenizer_key']!r}")
D3 = SM.load_dreaddit(CFG)                  # verifies the saved split file and keeps text-only columns
for k, v in D3.items():
    print(f"{k:<10} n={len(v):>5}  stress share={v['label'].mean():.3f}  columns={list(v.columns)}")
print("runs will be written to:", SM.runs_root(CFG))

In [ ]:
# Step 3.0b: LABEL MEANING CHECK. Notebook view only: scrubbed, truncated, 3 per label, never saved.
# The HF card only says `label: int64`. Confirm by reading: do label-1 texts express stress?
pd.set_option("display.max_colwidth", 200)
view = (D3["train"].groupby("label", group_keys=False)[list(D3["train"].columns)]
        .apply(lambda g: g.sample(3, random_state=SEED))[["label", "community", "text_clean"]])
view["text_clean"] = view["text_clean"].str.slice(0, 200)
display(view)

In [ ]:
# Step 3.0c: batch size for THIS laptop's device (session only; config.yaml is untouched). The EFFECTIVE batch stays 16
# (train_batch_size x gradient_accumulation_steps), so results are comparable between laptops; only memory per step changes.
import torch
from stress_signals.utils import select_device
DEVICE = select_device(CFG)
LOCAL_TRAIN_BATCH = "auto"        # "auto" | an int, e.g. 4 (if you hit "out of memory") | None: config.yaml settings
sm = CFG["stress_model"]
effective = sm["train_batch_size"] * sm["gradient_accumulation_steps"]
if LOCAL_TRAIN_BATCH == "auto":
    if DEVICE == "cuda":
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        LOCAL_TRAIN_BATCH = 16 if vram >= 14 else 8 if vram >= 7 else 4       # roberta-base, 256 tokens, fp16
    else:
        LOCAL_TRAIN_BATCH = None
if LOCAL_TRAIN_BATCH:
    LOCAL_TRAIN_BATCH = min(int(LOCAL_TRAIN_BATCH), effective)
    sm["train_batch_size"], sm["gradient_accumulation_steps"] = LOCAL_TRAIN_BATCH, max(1, effective // LOCAL_TRAIN_BATCH)
    sm["eval_batch_size"] = min(sm["eval_batch_size"], 2 * LOCAL_TRAIN_BATCH)
print(f"device {device_label(DEVICE)} | train batch {sm['train_batch_size']} x accumulation {sm['gradient_accumulation_steps']} "
      f"= effective {sm['train_batch_size'] * sm['gradient_accumulation_steps']} | eval batch {sm['eval_batch_size']} "
      f"| precision {'fp16' if DEVICE == 'cuda' else 'fp32'}")

### 3A. Baseline: TF-IDF (word 1-2 + char_wb 2-5) with logistic regression
This is a reference point. The transformer has to beat it by a clear margin, beyond the confidence intervals, to justify its cost. C is chosen on validation F1(stress).

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3A.1: fit the baseline ONCE (CPU by design, ~1-3 min). Skipped when it is already done.
BASE_DIR = SM.runs_root(CFG) / "baseline_tfidf_lr"
H.run_step(CFG, H.BASELINE_STEP, "Fit TF-IDF + logistic regression baseline", lambda: SM.fit_baseline(CFG, D3),
           done_marker=BASE_DIR / "DONE.json", skip_result=BASE_DIR)
if (BASE_DIR / "c_selection_validation.csv").exists():
    print(pd.read_csv(BASE_DIR / "c_selection_validation.csv").round(4))

In [ ]:
# Step 3A.2: baseline metrics ONCE (validation + test, threshold 0.5, cluster-bootstrap 95% CIs). Later: loaded from file.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["baseline"], "3A.2", R3["why_baseline"]):
    _f = BASE_DIR / "evaluation.json"
    BASE_EVAL = H.run_step(CFG, "3A.2/baseline_eval", "Evaluate the baseline", lambda: SM.evaluate_baseline(CFG),
                           done_marker=_f, skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")),
                           force="3A.2" in FORCE_RERUN)
    for split in ("validation", "test"):
        m, ci = BASE_EVAL[split]["all"], BASE_EVAL[split]["ci"]
        print(f"{split:<10} " + "  ".join(f"{k}={m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]"
                                          for k in ("f1_stress", "f1_macro", "roc_auc", "pr_auc", "accuracy")))

### 3B. Transformer fine-tuning
One run per seed, each resume-safe. Seeds that are done are skipped; an interrupted seed resumes from its last epoch checkpoint.
Speed depends on the laptop: a few minutes per epoch on a recent NVIDIA GPU, much longer on a small GPU, hours per seed on a CPU.

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3B.1: fine-tune every seed that is not done yet, on this laptop's best device.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if R3["seeds_missing"]:
    print(f"to train: seeds {R3['seeds_missing']} on {device_label(DEVICE)}")
    if DEVICE == "cpu":
        print("!!! NO GPU on this laptop: each seed takes HOURS on the CPU. Interrupt the kernel to stop; "
              "a re-run resumes from the last finished epoch.")
for seed in CFG["stress_model"]["seeds"]:
    rd = SM.run_dir_for(CFG, MODEL_CHOICE, seed)
    H.run_step(CFG, H.train_step_id(MODEL_CHOICE, seed), f"Fine-tune {MODEL_CHOICE}, seed {seed}",
               lambda seed=seed: SM.train_stress_seed(CFG, seed, MODEL_CHOICE), done_marker=rd / "DONE.json", skip_result=rd)
    if (rd / "train_summary.json").exists():
        summ = json.loads((rd / "train_summary.json").read_text())
        print(f"seed {seed}: best validation f1_stress = {summ.get('best_metric_validation_f1_stress')} "
              f"| epochs = {summ.get('epochs_completed')} | device = {summ.get('device')}")

In [ ]:
# Step 3B.2: training curves from the CSV logs (no training). Validation F1(stress) per epoch, per seed.
for seed, rd in SM.completed_seeds(CFG, MODEL_CHOICE).items():
    log = pd.read_csv(rd / "train_log.csv")
    ev = log[log["key"].isin(["eval_f1_stress", "eval_loss"])].pivot_table(index="epoch", columns="key", values="value")
    print(f"seed {seed}\n", ev.round(4))

### 3C. Evaluation (after training): validation and official test, uncalibrated at threshold 0.5
CIs come from a **cluster bootstrap over post_id** (1,000 resamples), because segments of the same post are not independent. *High-confidence* means
Dreaddit `confidence >= stress_model.high_confidence_threshold`. Reading these numbers: if the high-confidence F1 is much higher, many of the errors fall on
texts that annotators also disagreed about.

In [ ]:
# Step 3C.1: per-seed metrics + CIs, mean +/- std across seeds ONCE (reads saved logits). Later: loaded from file.
KEYS = ["accuracy", "precision_stress", "recall_stress", "f1_stress", "f1_macro", "roc_auc", "pr_auc"]

def _load_ev3(path):
    ev = json.loads(path.read_text(encoding="utf-8"))
    ev["seeds"] = {int(k): v for k, v in ev["seeds"].items()}        # JSON keys are strings
    return ev

R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.1", R3["why_seeds"]):
    _f = SM.runs_root(CFG) / MODEL_CHOICE / "evaluation_uncalibrated.json"
    EV3 = H.run_step(CFG, f"3C.1/evaluate_seeds/{MODEL_CHOICE}", "Evaluate all seeds (uncalibrated)",
                     lambda: SM.evaluate_runs(CFG, MODEL_CHOICE), done_marker=_f, skip_result=lambda: _load_ev3(_f),
                     force="3C.1" in FORCE_RERUN)
    for split in ("validation", "test"):
        rows = []
        for seed, r in EV3["seeds"].items():
            m, ci = r[split]["all"], r[split]["ci"]
            rows.append({"seed": seed, **{k: f"{m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]" for k in KEYS},
                         "confusion": m["confusion"]})
        print(f"\n== {split} (threshold 0.5, uncalibrated)"); display(pd.DataFrame(rows))
        print("mean +/- std across seeds:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3["summary"][split].items() if k in KEYS})

In [ ]:
# Step 3C.2: high-confidence subset (same metrics restricted to confident annotations; view of 3C.1)
if H.ready(R3["all_seeds"], "3C.2", R3["why_seeds"]):
    for split in ("validation", "test"):
        hc = {seed: r[split]["high_confidence"] for seed, r in EV3["seeds"].items()}
        first = next(iter(hc.values()))
        print(f"{split}: confidence >= {first['confidence_threshold']} keeps {first['share_of_split']:.1%} of rows (n={first['n']})")
        print("  mean +/- std:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3['summary'][f'{split}_high_confidence'].items() if k in KEYS})

In [ ]:
# Step 3C.3: final seed + per-subreddit error analysis (aggregates) + a small error view (scrubbed, truncated, never saved)
if H.ready(R3["all_seeds"], "3C.3", R3["why_seeds"]):
    FINAL_SEED = SM.select_final_seed(EV3)            # best VALIDATION f1_stress
    print("final seed (best validation f1_stress):", FINAL_SEED)
    display(SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE).round(3))
    te = SM.load_predictions(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED), "test").merge(
        D3["test"][["record_id", "text_clean"]], on="record_id")
    te["error"] = (te["p"] >= 0.5).astype(int) != te["label"]
    errs = te[te["error"]].assign(margin=lambda d: (d["p"] - 0.5).abs()).sort_values("margin", ascending=False)
    nshow = min(CFG["privacy"]["max_error_examples_in_notebook"], 10)
    view = errs.head(nshow)[["community", "label", "p", "confidence"]].assign(
        text=errs.head(nshow)["text_clean"].map(lambda t: scrub_text(t, CFG)[:180]))
    display(view)   # most confident mistakes: label noise, sarcasm, or topic shortcuts?

### 3D. Calibration and threshold (fitted on VALIDATION only), then the calibrated test report once
- **Temperature scaling** fits one scalar T that minimises validation NLL. It changes confidence but not the ranking, so ROC-AUC is unchanged.
- **Threshold:** search 0.40-0.60 for the best validation F1(stress). Apply it only if the gain over 0.5 is clear (gain >= `apply_rule.min_gain`
  and the paired-bootstrap CI is above 0). The gain is measured on the same validation set that chose the threshold, so it is optimistic; the minimum gain is the safety margin.
- **Reading ECE:** it is the average gap between predicted probability and observed frequency, so lower is better. Downstream, aggregate signals average calibrated probabilities,
  so calibration matters more for this project than the last point of F1.

In [ ]:
# Step 3D.1: fit T + threshold on validation ONCE; reliability diagram; ECE / Brier before vs after. Later: loaded from file.
if H.ready(R3["all_seeds"], "3D.1", R3["why_seeds"]):
    _f = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "calibration.json"
    CAL3 = H.run_step(CFG, f"3D.1/calibrate/{MODEL_CHOICE}/seed{FINAL_SEED}", "Calibrate the final seed on validation",
                      lambda: SM.calibrate(CFG, FINAL_SEED, MODEL_CHOICE), done_marker=_f,
                      skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")), force="3D.1" in FORCE_RERUN)
    dec = CAL3["threshold_decision"]; g = dec["validation_gain_f1_stress"]
    print(f"T = {CAL3['temperature']:.3f} | tuned threshold (searched {CAL3['threshold_search_grid'][:2]}) = {CAL3['threshold_tuned']:.2f}")
    print(f"validation F1(stress) gain over 0.5 = {g['gain']:+.4f} [95% CI {g['lo']:+.4f}, {g['hi']:+.4f}] "
          f"-> APPLIED threshold = {CAL3['threshold']:.2f} ({dec['reason']})")
    for k in ("before", "after"):
        v = CAL3[f"validation_{k}"]
        print(f"validation {k:<6}: ECE {v['ece']:.4f} | Brier {v['brier']:.4f} | F1(stress) {v['f1_stress']:.3f} @ {v['threshold']:.2f}")
    print(f"validation after, at tuned {CAL3['threshold_tuned']:.2f}: F1(stress) {CAL3['validation_after_tuned']['f1_stress']:.3f}")
    from IPython.display import Image
    display(Image(str(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "reliability_validation.png")))

In [ ]:
# Step 3D.2: FINAL calibrated test report: computed ONCE (FINAL_TEST_REPORT.json is reused afterwards).
# Reported at BOTH thresholds; "applied" is the one the decision rule picked on validation.
if H.ready(R3["all_seeds"], "3D.2", R3["why_seeds"]):
    _f = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "FINAL_TEST_REPORT.json"
    FIN3 = H.run_step(CFG, f"3D.2/final_test_report/{MODEL_CHOICE}/seed{FINAL_SEED}", "Final calibrated test report",
                      lambda: SM.final_test_report(CFG, FINAL_SEED, MODEL_CHOICE, force="3D.2" in FORCE_RERUN),
                      done_marker=_f, skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")),
                      force="3D.2" in FORCE_RERUN)
    print(f"TEST (T={FIN3['temperature']:.3f}); applied threshold = {FIN3['threshold']:.2f} [{FIN3['threshold_applied']}]: "
          f"{FIN3['threshold_decision_reason']}")
    rows = []
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        for k in KEYS + ["brier"]:
            rows.append({"threshold": f"{b['threshold']:.2f}", "metric": k, "value": round(b["test"][k], 3),
                         "ci95": f"[{b['test_ci'][k]['lo']:.3f}, {b['test_ci'][k]['hi']:.3f}]" if k in b["test_ci"] else ""})
    display(pd.DataFrame(rows).pivot(index="metric", columns="threshold", values=["value", "ci95"]))
    print("n =", FIN3["test"]["n"], "| ECE (threshold-free)", round(FIN3["test_ece"], 4))
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        print(f"  @{b['threshold']:.2f} confusion {b['test']['confusion']} | high-confidence F1(stress) "
              f"{b['test_high_confidence']['f1_stress']:.3f} (n={b['test_high_confidence']['n']})")

### 3E. Shortcut-learning checks
**(i) Topic masking (inference only).** Subreddit names and domain words (money, rent, abuse, boyfriend, ...) are replaced with the mask token.
A *control* condition masks the same number of random other words in each text, repeated over **5 random draws** (mean, sd and range are reported). How to read it:
- If the topic-masking drop is larger than **every** control draw's drop (`topic_drop_exceeds_all_control_draws`), the model leans on topic words, and its scores may reflect *what a
  community talks about* rather than how stressed the language is. That is a direct threat to comparisons between communities.
- If both drops are similar and small, the model mostly uses stress expression.
- Some drop is expected, because words like "anxiety" also carry real stress signal.

**(ii) Leave-one-subreddit-out (training, optional).** Train without one subreddit and test on all of its rows. How to read it: a large gap compared with in-domain
performance means the model does not generalise to unseen communities. That matters because the target corpus has communities Dreaddit lacks.

**(iii) Baseline top n-grams.** Look at the words with the largest weights. If the strongest stress words are topic words (`topic_term=True`) rather than
affect, coping or pressure words ("can't", "anxious", "scared", "overwhelmed"), the baseline is keying on topic.

In [ ]:
# Step 3E.1: topic masking vs random-masking control (5 draws) on the official test set ONCE (inference on this laptop's
# device). Uses the APPLIED threshold from 3D. Diagnostic only: nothing is tuned on these numbers. Later: loaded from file.
if H.ready(R3["all_seeds"], "3E.1", R3["why_seeds"]):
    model_dir = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "best_model"
    _f = SM.runs_root(CFG) / MODEL_CHOICE / "shortcut_masking.json"
    SHORT = H.run_step(CFG, f"3E.1/topic_masking/{MODEL_CHOICE}/seed{FINAL_SEED}", "Topic-masking shortcut check",
                       lambda: SM.shortcut_masking_eval(CFG, model_dir, "test", CAL3["temperature"], CAL3["threshold"], MODEL_CHOICE),
                       done_marker=_f, skip_result=lambda: pd.DataFrame(json.loads(_f.read_text(encoding="utf-8"))["results"]),
                       force="3E.1" in FORCE_RERUN)
    display(SHORT[["condition", "draw_seed", "f1_stress", "roc_auc", "accuracy", "predicted_positive_rate"]].round(3))
    print(f"texts with >=1 masked term: {SHORT['texts_with_masked_terms'].iloc[0]} of {SHORT['n'].iloc[0]}; "
          f"terms masked: {SHORT['masked_terms_total'].iloc[0]}")
    display(SM.summarise_masking(SHORT).round(4))

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3E.2 (OPTIONAL): leave-one-subreddit-out. Trains one model per held-out subreddit (distilroberta by default,
# 1 seed, config stress_model.shortcut.loso_*): ~10 extra trainings. Each subreddit runs ONCE; resume-safe like 3B.1.
RUN_LOSO = False      # True: train the LOSO models (Run All skips them while this is False)
LOSO_MODEL = CFG["stress_model"]["shortcut"]["loso_model"]
for sub in sorted(pd.concat(D3.values())["community"].unique()):
    d = SM.runs_root(CFG) / "loso" / f"{LOSO_MODEL}_{sub}"
    H.run_step(CFG, H.loso_step_id(LOSO_MODEL, sub), f"LOSO: train {LOSO_MODEL} without r/{sub}",
               lambda sub=sub: SM.train_loso(CFG, [sub])[sub], done_marker=d / "DONE.json", skip_result=d,
               allow=RUN_LOSO, blocked_reason="optional: set RUN_LOSO = True at the top of this cell")
LOSO = SM.evaluate_loso(CFG)
if not len(LOSO):
    print("no finished LOSO runs yet")
elif "FINAL_SEED" in globals():
    inorig = SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE)[["community", "n", "f1_stress", "roc_auc"]]
    display(LOSO.merge(inorig, left_on="heldout_subreddit", right_on="community", suffixes=("_heldout", "_in_domain_test")).round(3))
else:
    display(LOSO.round(3))

In [ ]:
# Step 3E.3: baseline n-grams with the largest weights (view; needs 3A.1). topic_term=True = contains a subreddit/domain term.
if H.ready(R3["baseline"], "3E.3", R3["why_baseline"]):
    NG = SM.top_ngrams(CFG)
    for direction in ("stress", "not_stress"):
        sub = NG[(NG["direction"] == direction) & (NG["type"] == "word")]
        print(f"\n{direction}: share of top word n-grams that are topic terms = {sub['topic_term'].mean():.2f}")
        display(sub[["rank", "ngram", "coef", "topic_term"]].head(20))

### 3F. Save the model bundle
This writes `trained_models/stress/v<YYYYMMDD>/` once. The bundle holds the weights and tokenizer,
`stress_config.json` (label map, max length, chunking rule, temperature, threshold, model name, seed), `metrics.json`, `model_card.md` and
`manifest.json` (SHA-256 of every file).

In [ ]:
# Step 3F: write the bundle ONCE (no training). Needs 3D.2; optionally 3A, 3E.1, 3E.2 so metrics.json includes them.
_ok = R3["all_seeds"] and "FINAL_SEED" in globals() and (SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "FINAL_TEST_REPORT.json").exists()
if H.ready(_ok, "3F", R3["why_seeds"]):
    def _bundle_from_log():
        ev = H.done_event(CFG, H.BUNDLE_STEP)
        return PROJECT_ROOT / ev["outputs"][0] if ev and ev["outputs"] else None
    BUNDLE3 = H.run_step(CFG, H.BUNDLE_STEP, f"Save stress model bundle ({MODEL_CHOICE}, seed {FINAL_SEED})",
                         lambda: SM.save_bundle(CFG, FINAL_SEED, MODEL_CHOICE, overwrite="3F" in FORCE_RERUN),
                         skip_result=_bundle_from_log, force="3F" in FORCE_RERUN)
    print("bundle:", BUNDLE3)
    if BUNDLE3 is not None and BUNDLE3.is_dir():
        for f in sorted(BUNDLE3.iterdir()):
            print(f"  {f.name:<28} {f.stat().st_size/1e6:8.2f} MB")
        print((BUNDLE3 / "model_card.md").read_text()[:1500])

### Handoff summary
What ran, on which laptop and device, when, and how long it took (also in `trained_models/HANDOFF.md`).

In [ ]:
# Handoff summary: one row per step (state, machine/device, time, duration)
H.show(CFG)

## Step 4: Emotion model on GoEmotions

_Not implemented yet. Filled in at Step 4._

## Step 5: Stressor taxonomy and stressor classifier

_Not implemented yet. Filled in at Step 5._

## Step 6: TensiStrength auxiliary signal and SenticNet external validation

_Not implemented yet. Filled in at Step 6._

## Step 7: Unified inference package and artifact bundle

_Not implemented yet. Filled in at Step 7._

## Step 8: Ingestion to a Parquet data lake (pilot, then main corpus)

_Not implemented yet. Filled in at Step 8._

## Step 9: PySpark distributed NLP inference

_Not implemented yet. Filled in at Step 9._

## Step 10: Spark analytics (temporal, emotions, stressors, topics)

_Not implemented yet. Filled in at Step 10._

## Step 11: Baseline and change detection -> collective pressure signals

_Not implemented yet. Filled in at Step 11._

## Step 12: PostgreSQL storage, aggregates only (optional)

_Not implemented yet. Filled in at Step 12._

## Step 13: Streamlit dashboard

_Not implemented yet. Filled in at Step 13._

## Step 14: Final evaluation report, ethics and documentation

_Not implemented yet. Filled in at Step 14._

## Step 15: Kafka + Spark Structured Streaming (optional)

_Not implemented yet. Filled in at Step 15._